# Twin-screw compressor model Kaufmann et al.

**Reference**

- Kaufmann, Irrgang, Schifflechner, Spliethoff (2024): *Fast and accurate
  modelling of twin-screw compressors: A generalised low-order approach*.
  Applied Thermal Engineering 257, 124238.
  https://doi.org/10.1016/j.applthermaleng.2024.124238


In [ ]:
# coefficients and reference
N_REF = 50.0

C_ETA_VOL = [
    0.946384, 0.509457, -0.083204, -1.647231, 0.236403, 0.001298,
    2.247307, -0.297059, -0.007651, 0.000305, -1.386582, 0.171104,
    0.007127, -0.000170, -0.000025, 0.317757, -0.036833, -0.002011,
    -0.000041, 0.000026
]
C_ETA_IS = [
    -83.1921, -4.7660, 528.7869, 2.8098, -7.6153, -629.7470,
    -4.9671, 30.6016, 320.7168, -2.8907, -8.5495, -62.5854
]
RVN_THRESHOLD = 1.3379


# polynomial evaluation methods
def eta_vol(x, y):
    # efficiency -> here dimensionless
    c = C_ETA_VOL
    return (
        c[0] + c[1] * x + c[2] * y + c[3] * x ** 2 + c[4] * x * y
        + c[5] * y ** 2 + c[6] * x ** 3 + c[7] * x ** 2 * y
        + c[8] * x * y ** 2 + c[9] * y ** 3 + c[10] * x ** 4
        + c[11] * x ** 3 * y + c[12] * x ** 2 * y ** 2 + c[13] * x * y ** 3
        + c[14] * y ** 4 + c[15] * x ** 5 + c[16] * x ** 4 * y
        + c[17] * x ** 3 * y ** 2 + c[18] * x ** 2 * y ** 3
        + c[19] * x * y ** 4
    )


def _eta_is_poly(x, y):
    c = C_ETA_IS
    return (
        c[0] + c[1] * x + c[2] * y + c[3] * x ** 2 + c[4] * x * y
        + c[5] * y ** 2 + c[6] * x ** 2 * y + c[7] * x * y ** 2
        + c[8] * y ** 3 + c[9] * x ** 2 * y ** 2 + c[10] * x * y ** 3
        + c[11] * y ** 4
    )


def _deta_is_poly_dy(x, y):
    c = C_ETA_IS
    return (
        c[2] + c[4] * x + 2 * c[5] * y + c[6] * x ** 2 + 2 * c[7] * x * y
        + 3 * c[8] * y ** 2 + 2 * c[9] * x ** 2 * y + 3 * c[10] * x * y ** 2
        + 4 * c[11] * y ** 3
    )


def eta_is_percent(x, rvn):
    # efficiency in %
    if rvn <= RVN_THRESHOLD:
        return _eta_is_poly(x, rvn)
    return (
        _eta_is_poly(x, RVN_THRESHOLD)
        + _deta_is_poly_dy(x, RVN_THRESHOLD) * (rvn - RVN_THRESHOLD)
    )